# Face Recognition Attendance Data – Cleaning & Visualization

This project demonstrates data cleaning, preprocessing, analysis and visualization of attendance records generated from a face-recognition attendance system.

## Objectives
- Handle missing values and duplicates
- Standardize dates, times, names and status values
- Calculate student and daily attendance percentages
- Identify low-attendance students
- Visualize attendance patterns using Matplotlib

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('attendance_raw.csv')
df.head()

In [ ]:
# Inspect data quality
print(df.info())
print('\nMissing values:\n', df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())

In [ ]:
# Clean names, department and status
df['Name'] = df['Name'].fillna('Unknown').astype(str).str.strip()
df['Department'] = df['Department'].astype(str).str.replace('AI & DS', 'AI&DS', regex=False).str.strip()
df['Status'] = df['Status'].astype(str).str.strip().str.title()

# Convert date and time fields
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df['Time'] = pd.to_datetime(df['Time'], format='%H:%M', errors='coerce').dt.strftime('%H:%M')

# Remove duplicate attendance entries
df = df.drop_duplicates(subset=['Student_ID', 'Date'], keep='first')
df = df[df['Name'] != 'Unknown']
df = df[df['Status'].isin(['Present', 'Late', 'Absent'])]
df = df.sort_values(['Date', 'Student_ID']).reset_index(drop=True)

df.to_csv('attendance_cleaned.csv', index=False)
df.head()

In [ ]:
# Basic KPIs
total_students = df['Student_ID'].nunique()
total_records = len(df)
present = (df['Status'] == 'Present').sum()
late = (df['Status'] == 'Late').sum()
absent = (df['Status'] == 'Absent').sum()
attendance_rate = (present + late) / total_records * 100

print('Total Students:', total_students)
print('Total Attendance Records:', total_records)
print('Present:', present)
print('Late:', late)
print('Absent:', absent)
print(f'Overall Attendance Rate: {attendance_rate:.2f}%')

In [ ]:
# Student-wise attendance analysis
student_summary = df.groupby(['Student_ID', 'Name'], as_index=False).agg(
    Total_Days=('Date', 'count'),
    Present=('Status', lambda x: (x == 'Present').sum()),
    Late=('Status', lambda x: (x == 'Late').sum()),
    Absent=('Status', lambda x: (x == 'Absent').sum())
)
student_summary['Attendance_Percentage'] = ((student_summary['Present'] + student_summary['Late']) / student_summary['Total_Days'] * 100).round(2)
student_summary.sort_values('Attendance_Percentage', ascending=False).head(10)

In [ ]:
# Identify students below 75% attendance
low_attendance = student_summary[student_summary['Attendance_Percentage'] < 75]
low_attendance[['Student_ID', 'Name', 'Attendance_Percentage']]

In [ ]:
# Daily attendance trend
daily = df.groupby('Date').agg(
    Present=('Status', lambda x: (x == 'Present').sum()),
    Late=('Status', lambda x: (x == 'Late').sum()),
    Absent=('Status', lambda x: (x == 'Absent').sum())
).reset_index()
daily['Attendance_Percentage'] = ((daily['Present'] + daily['Late']) / total_students * 100).round(2)

plt.figure(figsize=(10,5))
plt.plot(daily['Date'], daily['Attendance_Percentage'], marker='o')
plt.title('Daily Attendance Percentage')
plt.xlabel('Date')
plt.ylabel('Attendance %')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Status distribution
status_counts = df['Status'].value_counts().reindex(['Present', 'Late', 'Absent']).fillna(0)
plt.figure(figsize=(8,5))
status_counts.plot(kind='bar')
plt.title('Attendance Status Distribution')
plt.xlabel('Status')
plt.ylabel('Number of Records')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Student attendance chart
plot_students = student_summary.sort_values('Attendance_Percentage', ascending=True)
plt.figure(figsize=(10,7))
plt.barh(plot_students['Name'], plot_students['Attendance_Percentage'])
plt.title('Student Attendance Percentage')
plt.xlabel('Attendance %')
plt.ylabel('Student')
plt.tight_layout()
plt.show()

In [ ]:
# Department comparison
dept = df.groupby('Department').agg(
    Total_Records=('Status','count'),
    Present=('Status', lambda x: (x == 'Present').sum()),
    Late=('Status', lambda x: (x == 'Late').sum()),
    Absent=('Status', lambda x: (x == 'Absent').sum())
).reset_index()
dept['Attendance_Percentage'] = ((dept['Present'] + dept['Late']) / dept['Total_Records'] * 100).round(2)

plt.figure(figsize=(8,5))
plt.bar(dept['Department'], dept['Attendance_Percentage'])
plt.title('Attendance Percentage by Department')
plt.xlabel('Department')
plt.ylabel('Attendance %')
plt.tight_layout()
plt.show()

dept

## Key Findings
1. Duplicate records were identified and removed.
2. Missing/invalid values were handled during preprocessing.
3. Attendance status values were standardized to Present, Late and Absent.
4. Student-wise attendance percentages were calculated.
5. Daily and department-level attendance trends were visualized.

## Conclusion
The cleaned attendance dataset can be used for reporting and monitoring student attendance. The same workflow can be connected to the database/API of a real face-recognition attendance system.